In [1]:
from ultralytics import YOLO
import cv2
import numpy as np
import torch
import os
from tqdm import tqdm
data_collection = 'cha_short'

# Get all video files from inputs folder
def get_video_files(input_dir=f'inputs/{data_collection}', suffix=None):
    video_extensions = ('.mp4', '.MP4', '.avi', '.AVI', '.mov', '.MOV')
    video_files = []
    for file in os.listdir(input_dir):
        if file.endswith(video_extensions):
            video_name = os.path.splitext(file)[0]
            if suffix is None or video_name.endswith(suffix):
                video_files.append({
                    'name': video_name,
                    'path': os.path.join(input_dir, file)
                })
    return video_files

# Input settings
resolution = (3840, 2160)
show = True
save = True

In [2]:
# Load the YOLO model
model_person = YOLO('checkpoints/yolo/yolo11l.pt')
model = YOLO('checkpoints/yolo/rohan_pretrained.pt')

# Configure model settings
model.conf = 0.5  # NMS confidence threshold
model.iou = 0.5   # NMS IoU threshold

In [3]:
def crop_img(img, box):
    """Crop image based on bounding box"""
    x1, y1, x2, y2 = box
    return img[int(y1):int(y2), int(x1):int(x2)]

def detect_hands_in_frame(frame):
    """Detect hands in a single frame using YOLO model"""
    # Process frame with YOLO
    results = model_person(frame, classes=0)
    # print(results)
    boxes = []
    confidence = []
    
    # Process detections
    for r in results:
        boxes_tensor = r.boxes.xyxy.cpu()
        confs = r.boxes.conf.cpu()
        for box1, conf in zip(boxes_tensor, confs):
            cropped = crop_img(frame, box1)
            results_hands = model(cropped)
            for r in results_hands:
                boxes_tensor = r.boxes.xyxy.cpu()
                confs = r.boxes.conf.cpu()
                for box2, conf in zip(boxes_tensor, confs):
                    if conf > model.conf:
                        # print(box2)
                        adjusted_box = np.add(np.array(box2).reshape(2, 2), box1[:2])
                        boxes.append(adjusted_box.flatten())
                        confidence.append(conf)
                       
                
    return np.array(boxes) if boxes else np.array([]), np.array(confidence) if confidence else np.array([])

In [4]:
def process_video(input_file, write_folder=True, show=True):
    """Process video file for hand detection"""
    cap = cv2.VideoCapture(input_file)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video file: {input_file}")
    
    frame_idx = 0
    output_boxes = []
    out_frame_idx = None
    
    # Setup output directory for frames
    if write_folder:
        parentdir = os.path.dirname(input_file)
        base_name = os.path.splitext(os.path.basename(input_file))[0]
        frames_dir = os.path.join(parentdir, base_name)
        os.makedirs(frames_dir, exist_ok=True)
    
    success = True
    while cap.isOpened() and success:   
        success, frame = cap.read()
        if not success:
            break
            
        # Save frame if requested
        if write_folder:
            output_file = os.path.join(frames_dir, f"{frame_idx:05d}.jpg")
            cv2.imwrite(output_file, frame)
            
        # Detect hands
        boxes, confidence = detect_hands_in_frame(frame)
        
        if len(boxes) > 0:
            output_boxes.append((boxes, frame_idx))
            
        if show:
            # Visualize detections
            for box, conf in zip(boxes, confidence):
                x1, y1, x2, y2 = map(int, box)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 255), 2)
                cv2.putText(frame, f"Hand, conf: {round(100*conf, 1)}%", (x1 - 30, y1 - 30), cv2.FONT_HERSHEY_PLAIN, 2, (255, 0, 255), 2)
            
            cv2.namedWindow('Detections', cv2.WINDOW_NORMAL)
            cv2.resizeWindow('Detections', 1920, 1080)
            cv2.imshow('Detections', frame)
            if cv2.waitKey(1) == 113:
                success = False

                
        frame_idx += 1
        print(f"Processing frame {frame_idx}", end='\r')
    
    cap.release()
    if show:
        cv2.destroyAllWindows()
        
    return output_boxes

In [5]:
def save_to_file(boxes, output_file):
    
    # Save output to file
    with open(output_file, 'w+') as f:
        for boxes, frame_idx in boxes:
            f.write(f"{frame_idx} ")
            for box in boxes:
                f.write(f"{box[0]} {box[1]} {box[2]} {box[3]} ")
            f.write("\n")


In [6]:
def group_bounding_boxes(bbox_frames, iou_threshold=0.5, frame_gap_threshold=60):
    """
    Group bounding boxes across frames that likely belong to the same object
    
    Args:
        bbox_frames: List of tuples (bboxes, frame_idx) where bboxes is an array of shape (N, 4)
        iou_threshold: Threshold for IoU to consider boxes as matching
        frame_gap_threshold: Maximum number of frames between detections to consider them as the same object
    
    Returns:
        List of lists where each inner list contains tuples of (bbox, frame_idx) belonging to the same object
    """
    
    def calculate_iou(box1, box2):
        # Calculate intersection over union between two boxes
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        
        intersection = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
        area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
        
        return intersection / (area1 + area2 - intersection)
    
    # Initialize object tracks
    tracks = []
    
    # Process each frame's detections
    for bboxes, frame_idx in bbox_frames:
        # Convert single bbox to list if necessary
        if len(bboxes.shape) == 1:
            bboxes = bboxes.reshape(1, -1)
            
        for bbox in bboxes:
            matched = False
            
            # Try to match with existing tracks
            for track in tracks:
                last_bbox, last_frame = track[-1]
                
                # Check if the frame gap is reasonable
                if frame_idx - last_frame > frame_gap_threshold:
                    continue
                
                # Calculate IoU with the last box in the track
                iou = calculate_iou(bbox, last_bbox)
                
                if iou >= iou_threshold:
                    # Add to existing track
                    track.append((bbox, frame_idx))
                    matched = True
                    break
            
            if not matched:
                # Start new track
                tracks.append([(bbox, frame_idx)])
    
    return tracks

In [8]:
# Process all videos in the inputs folder
video_files = get_video_files(suffix='synced_cut')
print(f"Found {len(video_files)} videos to process")

for video in video_files:
    print(f"\nProcessing video: {video['name']}")
    
    # Process the video
    bboxes = process_video(video['path'], write_folder=False, show=show)
    
    if len(bboxes) == 0:
        print("No hands detected")
    else:
        (boxes, frame_idxs) = zip(*bboxes)
        print(f"\nFound hands at frame {frame_idxs}")
        print(f"Bounding boxes: \n{boxes}")

        if save:
            os.makedirs(f"outputs/{data_collection}", exist_ok=True)
            save_to_file(bboxes, f"outputs/{data_collection}/{video['name']}.txt")

        print("Grouping bounding boxes...")
        tracks = group_bounding_boxes(bboxes)
        print(f"Found {len(tracks)} tracks")
        print(tracks)

Found 13 videos to process

Processing video: gopro10_synced_cut

0: 384x640 1 person, 216.6ms
Speed: 5.0ms preprocess, 216.6ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 640x640 (no detections), 57.8ms
Speed: 6.0ms preprocess, 57.8ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)
Processing frame 1
0: 384x640 1 person, 23.9ms
Speed: 2.0ms preprocess, 23.9ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 640x640 (no detections), 52.3ms
Speed: 0.0ms preprocess, 52.3ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 640)
Processing frame 2
0: 384x640 1 person, 33.7ms
Speed: 0.0ms preprocess, 33.7ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 640x640 1 hand, 45.7ms
Speed: 0.0ms preprocess, 45.7ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)
Processing frame 3
0: 384x640 1 person, 15.4ms
Speed: 13.4ms preprocess, 15.4ms inference, 0.0ms postprocess per image at shape (